# Phase 3 · Ensemble of finished runs
Loads each run's `best.pt` (it stores its own config `C`), predicts dev + test with it, averages the **softmax probabilities** (optionally weighted), reports dev macro-F1 for each member and for the ensemble, and writes one `submission.zip`.

**Setup:** every run folder listed in `RUN_DIRS` must be reachable from this Drive session (copy or add shortcuts for runs trained under other accounts). Only `best.pt` is needed from each. `rr_lib.py` must be the latest version on Drive.

In [ ]:
# 1 · Settings
from pathlib import Path
ACTION   = 'Train'          # Inspect | Train | Predict   (Predict re-writes the submission from best.pt)
PRESET   = 'MLA-prenorm'    # GQA-prenorm | MHA-prenorm | MHA-postnorm | MLA-prenorm
METHOD   = 'lora'           # full | partial | lora | rslora | lorma | ia3 | prefix | kd
RUN_ID   = 'mla-lora-pos-001'
CONTEXT  = 'window'         # window | sentence
CLASS_WEIGHT = 'sqrt'       # none | sqrt | inverse
EPOCHS, SEED = 6, 17
USE_POS = True   # document-position features (new in rr_lib.py v2)

COURSE_ROOT = Path('/content/drive/MyDrive/legal-llm-course')
PHASE_DIR   = COURSE_ROOT/'Phase_3_RR'
RUNS_ROOT   = COURSE_ROOT/'runs/phase3_rr'
BUNDLE_ZIP  = COURSE_ROOT/'phase2_lner_multimethod_bundle.zip'   # existing Phase 2 bundle: legal_scaffold, training_methods, tokenizer
RR_LIB      = COURSE_ROOT/'rr_lib.py'                            # Phase 3 helper, upload next to the bundle

# adapter / method knobs
RANK, ALPHA = 8, 16                 # lora / rslora / lorma
LORA_TARGETS = ('query', 'value')   # mapped per architecture (MLA -> query_up/value_up)
PARTIAL_LAYERS = (10, 11)
PREFIX_LEN = 16
# knowledge distillation
KD_TEACHER_RUN = RUNS_ROOT/'MHA-prenorm/full/mha-full-win-001'   # finished `full` run used as teacher
KD_STUDENT_LAYERS = (0, 2, 4, 6, 8, 10)                            # 12 -> 6 blocks, copied from Phase 1
KD_TEMPERATURE, KD_ALPHA = 2.0, 0.5

DEFAULT_LR = {  # backbone / adapter / head learning rates
 'full': (5e-5, 0, 1e-3), 'kd': (5e-5, 0, 1e-3), 'partial': (1e-4, 0, 1e-3),
 'lora': (0, 1e-3, 1e-3), 'rslora': (0, 1e-3, 1e-3), 'lorma': (0, 1e-3, 1e-3),
 'ia3': (0, 3e-3, 1e-3), 'prefix': (0, 3e-3, 1e-3)}
assert ACTION in ('Inspect','Train','Predict') and METHOD in DEFAULT_LR and CONTEXT in ('window','sentence')

PRESETS = {
 'GQA-prenorm': dict(vocab_size=32768,d_model=512,layers=12,heads=8,kv_heads=2,ffn_hidden=2048,context_length=2048,attention='gqa',norm_placement='pre',tie_embeddings=True,rope_base=10000,norm_epsilon=1e-6,init_std=.02),
 'MHA-prenorm': dict(vocab_size=32768,d_model=512,layers=12,heads=8,kv_heads=8,ffn_hidden=2048,context_length=2048,attention='mha',norm_placement='pre',tie_embeddings=True,rope_base=10000,norm_epsilon=1e-6,init_std=.02),
 'MHA-postnorm': dict(vocab_size=32768,d_model=512,layers=12,heads=8,kv_heads=8,ffn_hidden=2048,context_length=2048,attention='mha',norm_placement='post',tie_embeddings=True,rope_base=10000,norm_epsilon=1e-6,init_std=.02),
 'MLA-prenorm': dict(vocab_size=32768,d_model=512,layers=12,heads=8,kv_heads=2,ffn_hidden=2048,context_length=2048,attention='mla',norm_placement='pre',tie_embeddings=True,rope_base=10000,norm_epsilon=1e-6,init_std=.02,mla_latent=256,mla_query_rank=256,mla_rope_dim=32)}
# NOTE: presets copied from the Phase 2 notebook (same Phase 1 snapshots, same tokenizer).

# ---- ensemble settings -------------------------------------------------------------
R3 = RUNS_ROOT
RUN_DIRS = [   # edit: any finished run folders (each must contain best.pt)
    R3/'MLA-prenorm/lora/mla-lora-win-001',
    R3/'GQA-prenorm/lorma/gqa-lorma-win-001',
    R3/'MHA-prenorm/ia3/mha-ia3-win-001',
    R3/'MHA-prenorm/full/mha-full-win-001',
    # R3/'MLA-prenorm/lora/mla-lora-pos-001',
    # R3/'MLA-prenorm/full/mla-full-pos-001',
]
WEIGHTS = None          # None = equal weights, or e.g. [1,1,1,2]
OUT_DIR = RUNS_ROOT/'ensemble'/'ens-001'

In [ ]:
# 2 · Drive, helper bundle, tokenizer, device
from google.colab import drive
drive.mount('/content/drive')
import sys, shutil, zipfile, hashlib, json, subprocess, importlib.util, random
LOCAL = Path('/content/phase3-rr')
if LOCAL.exists(): shutil.rmtree(LOCAL)
LOCAL.mkdir(parents=True)
with zipfile.ZipFile(BUNDLE_ZIP) as z: z.extractall(LOCAL)
shutil.copy(RR_LIB, LOCAL/'rr_lib.py')
sys.path.insert(0, str(LOCAL))
need = [p for p in ('tokenizers','huggingface_hub','scikit-learn') if importlib.util.find_spec(p.replace('scikit-learn','sklearn')) is None]
if need: subprocess.check_call([sys.executable,'-m','pip','install','-q',*need])
import torch, rr_lib as R
from tokenizers import Tokenizer
from training_methods import resolve_snapshot_checkpoint
TOKENIZER_SHA256 = '906194848c1632431aac7d186e7c4f49da83cb6cbd480fa222c8472401cbf8ab'
tp = LOCAL/'tokenizer/tokenizer.json'
assert hashlib.sha256(tp.read_bytes()).hexdigest() == TOKENIZER_SHA256, 'tokenizer differs from Phase 1/2'
TOK = Tokenizer.from_file(str(tp))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
random.seed(SEED); torch.manual_seed(SEED)
print('device', device, torch.cuda.get_device_name(0) if device.type=='cuda' else '')

In [ ]:
# 3 · Data (downloads Phase_3_RR from the gated HF dataset if not already on Drive)
from google.colab import userdata
try: token = userdata.get('HF_TOKEN')
except Exception: token = None
R.ensure_files(COURSE_ROOT, token)
LABELS = R.load_labels(PHASE_DIR/'Train/label_vocab.json')
L2I = {l:i for i,l in enumerate(LABELS)}
DATA = R.load_all(PHASE_DIR, LABELS)
for s in DATA.values(): R.tokenize_docs(s, TOK, 64)
print(len(LABELS), 'labels:', LABELS)
print(json.dumps(R.summarize(DATA, LABELS), indent=1))
ids = [d['id'] for d in DATA['test']]
print('test ids unique across CL+IT:', len(ids)==len(set(ids)), '| test sentences:', sum(len(d['text']) for d in DATA['test']))
print('example:', DATA['train'][0]['text'][:2], DATA['train'][0]['labels'][:2])

In [ ]:
PHASE1_OVERRIDE = {'MHA-prenorm': str(COURSE_ROOT/'runs/mha_pre/run-001/legal-token-stages-v1/run-001-milestone-60762-aa4db459/state.pt')}   # same snapshot Phase 2 used; mha_pre is a Drive shortcut so auto-search misses it

def _scan_snapshots(preset):
    """Diagnostic: list every snapshot.json under runs/ (follows Drive shortcuts) vs the wanted config."""
    import os
    from legal_scaffold.models import ModelConfig
    want = ModelConfig(**PRESETS[preset]).validate().__dict__
    rows = []
    for dp, dn, fn in os.walk(COURSE_ROOT/'runs', followlinks=True):
        if 'snapshot.json' in fn:
            try:
                m = json.loads((Path(dp)/'snapshot.json').read_text())
                ident = m.get('identity', {}); cfg = ident.get('model', ident.get('config')) or {}
                got = ModelConfig(**cfg).validate().__dict__ if cfg else {}
                st = m.get('state_inventory', {}).get('model')
                rows.append((got == want, m.get('stage'), m.get('progress', {}).get('step'), bool(st and (Path(dp)/st).is_file()), dp))
            except Exception as e:
                rows.append((False, 'ERR '+str(e)[:40], None, False, dp))
    return rows

def phase1_checkpoint(preset):
    if preset in PHASE1_OVERRIDE:
        return Path(PHASE1_OVERRIDE[preset]), {'stage': 'override'}
    for kw in (dict(stages=('legal_continued','wikitext_only'), tokenizer_sha256=TOKENIZER_SHA256),
               dict(stages=('legal_continued','wikitext_only'), tokenizer_sha256=None),
               dict(stages=('legal_continued','legal_only','wikitext_only','legal','pretrain'), tokenizer_sha256=None)):
        try:
            return resolve_snapshot_checkpoint(COURSE_ROOT/'runs', PRESETS[preset], **kw)
        except FileNotFoundError:
            pass
    rows = _scan_snapshots(preset)
    print(f'--- No usable Phase 1 snapshot for {preset}. {len(rows)} snapshot.json files found under runs/ ---')
    for r in sorted(rows, key=lambda r: (not r[0], str(r[1])))[:25]:
        print('config_match=%s stage=%s step=%s payload_present=%s  %s' % r)
    raise FileNotFoundError(f'{preset}: no snapshot with matching config + payload; set PHASE1_OVERRIDE[{preset!r}] to its state.pt')


def build(C):
    ck, man = phase1_checkpoint(C['preset'])
    m = R.build_model(PRESETS[C['preset']], ck, len(LABELS), 'full' if C['method']=='kd' else C['method'],
                      head_kw=C['head_kw'], adapter_kw=C['adapter_kw'] if C['method'] not in ('full','kd','partial') else {},
                      prefix_len=C['prefix_len'], layer_keep=C['layer_keep'])
    return m, ck



In [ ]:
# Predict dev + test with every member (one model in memory at a time)
import gc, torch.nn.functional as F
ids_dev = {d['id'] for d in DATA['dev']}
members = []
for rd in RUN_DIRS:
    saved = torch.load(Path(rd)/'best.pt', map_location='cpu', weights_only=False)
    C = saved['C']
    model, _ = build(C)
    missing = model.load_state_dict(saved['trainable'], strict=False)
    model.to(device).eval()
    wcfg = dict(core_max=C['core_max'], ctx_left=C['ctx_left'], ctx_right=C['ctx_right'], budget=C['budget'])
    amp = torch.bfloat16 if (device.type=='cuda' and torch.cuda.is_bf16_supported()) else torch.float16
    out = {}
    for split in ('dev','test'):
        _, lg = R.predict_docs(model, DATA[split], wcfg, L2I, device, C['eval_batch'], amp)
        out[split] = {k: F.softmax(torch.stack(v), -1) for k, v in lg.items()}
    members.append({'run': Path(rd).name, 'C': C, **out})
    print('loaded', Path(rd).name, '| saved dev F1', round(saved['dev']['macro_f1_present'],4))
    del model; gc.collect(); torch.cuda.empty_cache()

def dev_scores(prob_by_doc):
    gold, pred = [], []
    for d in DATA['dev']:
        gold += [L2I[l] for l in d['labels']]; pred += prob_by_doc[d['id']].argmax(-1).tolist()
    return R.macro_scores(gold, pred, LABELS)

def blend(idx, weights=None):
    w = weights or [1.0]*len(idx); tot = sum(w)
    return {split: {k: sum(wi*members[i][split][k] for wi, i in zip(w, idx))/tot for k in members[idx[0]][split]}
            for split in ('dev','test')}

import pandas as pd
rows = [{'members': m['run'], 'dev_F1': round(dev_scores(m['dev'])['macro_f1_present'],4),
         'dev_F1_all13': round(dev_scores(m['dev'])['macro_f1_all13'],4)} for m in members]
ens = blend(list(range(len(members))), WEIGHTS)
s = dev_scores(ens['dev'])
rows.append({'members': 'ENSEMBLE (all)', 'dev_F1': round(s['macro_f1_present'],4), 'dev_F1_all13': round(s['macro_f1_all13'],4)})
display(pd.DataFrame(rows))

In [ ]:
# Write the ensemble submission (all members). To use a subset, change idx below.
idx = list(range(len(members)))
final = blend(idx, WEIGHTS)
test_preds = {k: v.argmax(-1).tolist() for k, v in final['test'].items()}
SUB = OUT_DIR/'submission'
path = R.write_submission(test_preds, DATA['test'], LABELS, SUB)
chk = json.loads(path.read_text())
assert set(chk) == {d['id'] for d in DATA['test']}
assert all(len(chk[d['id']]) == len(d['text']) for d in DATA['test'])
with zipfile.ZipFile(SUB/'submission.zip') as z: assert z.namelist() == ['prediction.json']
print('wrote', SUB/'submission.zip')
s = dev_scores(blend(idx, WEIGHTS)['dev'])
print('ensemble dev macro-F1', round(s['macro_f1_present'],4), '| per-class F1:', {k: round(v['f1'],2) for k,v in s['per_class'].items()})